In [1]:
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import warnings

warnings.filterwarnings('ignore')

ROUND_TO_MILLISECOND = True

In [2]:
def get_cov(in_prefix, out_prefix):
    return {f"field.{in_prefix}{i}": f"{out_prefix}_{i}" for i in range(9)}

COMMON_HDR = {
    "%time": "recording_timestamp",
    "field.header.seq": "sequence_number",
    "field.header.stamp": "sensor_timestamp",
    "field.header.frame_id": "frame_id",
    "seq": "sequence_number",  
    "stamp": "sensor_timestamp",  
    "frame_id": "frame_id"    
}

COLUMN_MAPPING = {
    # 🔴 NHÓM 1: TRẠNG THÁI SỰ CỐ
    "failure_status-engines": {"%time": "recording_timestamp", "field.data": "engine_failure_status"},
    "failure_status-aileron": {"%time": "recording_timestamp", "field.data": "aileron_failure_status"},
    "failure_status-rudder": {"%time": "recording_timestamp", "field.data": "rudder_failure_status"},
    "failure_status-elevator": {"%time": "recording_timestamp", "field.data": "elevator_failure_status"},

    # 🟡 NHÓM 2: ĐIỀU HƯỚNG VÀ SAI SỐ
    "mavros-nav_info-roll": {**COMMON_HDR, "field.commanded": "commanded_roll", "field.measured": "measured_roll"},
    "mavros-nav_info-pitch": {**COMMON_HDR, "field.commanded": "commanded_pitch", "field.measured": "measured_pitch"},
    "mavros-nav_info-yaw": {**COMMON_HDR, "field.commanded": "commanded_yaw", "field.measured": "measured_yaw"},
    "mavros-nav_info-airspeed": {**COMMON_HDR, "field.commanded": "commanded_airspeed", "field.measured": "measured_airspeed"},
    "mavros-nav_info-velocity": {**COMMON_HDR, "field.coordinate_frame": "coordinate_frame", 
        "field.des_x": "desired_velocity_x", "field.des_y": "desired_velocity_y", "field.des_z": "desired_velocity_z",
        "field.meas_x": "measured_velocity_x", "field.meas_y": "measured_velocity_y", "field.meas_z": "measured_velocity_z"},
    "mavros-nav_info-errors": {**COMMON_HDR, "field.alt_error": "altitude_error", "field.aspd_error": "airspeed_error", 
        "field.xtrack_error": "crosstrack_error", "field.wp_dist": "waypoint_distance"},
    "mavctrl-rpy": {"%time": "recording_timestamp", "field.x": "measured_roll_rad", "field.y": "measured_pitch_rad", "field.z": "measured_yaw_rad"},
    "mavctrl-path_dev": {"%time": "recording_timestamp", "field.x": "along_track_deviation", "field.y": "cross_track_deviation", "field.z": "vertical_track_deviation"},

    # 🔵 NHÓM 3: ĐỊNH VỊ VÀ VỊ TRÍ
    "mavros-local_position-pose": {**COMMON_HDR, "field.pose.position.x": "local_position_x", "field.pose.position.y": "local_position_y", "field.pose.position.z": "local_position_z",
        "field.pose.orientation.x": "orientation_qx", "field.pose.orientation.y": "orientation_qy", "field.pose.orientation.z": "orientation_qz", "field.pose.orientation.w": "orientation_qw"},
    "mavros-local_position-odom": {**COMMON_HDR, "field.child_frame_id": "child_frame_id", 
        "field.pose.pose.position.x": "odom_position_x", "field.pose.pose.position.y": "odom_position_y", "field.pose.pose.position.z": "odom_position_z",
        "field.pose.pose.orientation.x": "odom_orientation_qx", "field.pose.pose.orientation.y": "odom_orientation_qy", "field.pose.pose.orientation.z": "odom_orientation_qz", "field.pose.pose.orientation.w": "odom_orientation_qw",
        "field.twist.twist.linear.x": "body_linear_velocity_x", "field.twist.twist.linear.y": "body_linear_velocity_y", "field.twist.twist.linear.z": "body_linear_velocity_z",
        "field.twist.twist.angular.x": "body_angular_velocity_x", "field.twist.twist.angular.y": "body_angular_velocity_y", "field.twist.twist.angular.z": "body_angular_velocity_z"},
    "mavros-local_position-velocity": {**COMMON_HDR, "field.twist.linear.x": "local_linear_velocity_x", "field.twist.linear.y": "local_linear_velocity_y", "field.twist.linear.z": "local_linear_velocity_z",
        "field.twist.angular.x": "body_angular_velocity_x", "field.twist.angular.y": "body_angular_velocity_y", "field.twist.angular.z": "body_angular_velocity_z"},
    "mavros-global_position-global": {**COMMON_HDR, "field.status.status": "gps_status", "field.status.service": "gps_service", "field.latitude": "latitude", "field.longitude": "longitude", "field.altitude": "gps_altitude",
        **get_cov("position_covariance", "gps_position_covariance"), "field.position_covariance_type": "gps_position_covariance_type"},
    "mavros-global_position-local": {**COMMON_HDR, "field.child_frame_id": "child_frame_id", "field.pose.pose.position.x": "gps_local_position_x", "field.pose.pose.position.y": "gps_local_position_y", "field.pose.pose.position.z": "gps_local_position_z",
        "field.pose.pose.orientation.x": "gps_local_orientation_qx", "field.pose.pose.orientation.y": "gps_local_orientation_qy", "field.pose.pose.orientation.z": "gps_local_orientation_qz", "field.pose.pose.orientation.w": "gps_local_orientation_qw",
        "field.twist.twist.linear.x": "gps_local_linear_velocity_x", "field.twist.twist.linear.y": "gps_local_linear_velocity_y", "field.twist.twist.linear.z": "gps_local_linear_velocity_z",
        "field.twist.twist.angular.x": "gps_local_angular_velocity_x", "field.twist.twist.angular.y": "gps_local_angular_velocity_y", "field.twist.twist.angular.z": "gps_local_angular_velocity_z"},
    "mavros-global_position-rel_alt": {"%time": "recording_timestamp", "field.data": "relative_altitude"},
    "mavros-global_position-compass_hdg": {"%time": "recording_timestamp", "field.data": "compass_heading"},
    "mavros-setpoint_raw-local": {**COMMON_HDR, "field.coordinate_frame": "coordinate_frame", "field.type_mask": "type_mask",
        "field.position.x": "target_position_x", "field.position.y": "target_position_y", "field.position.z": "target_position_z",
        "field.velocity.x": "target_velocity_x", "field.velocity.y": "target_velocity_y", "field.velocity.z": "target_velocity_z",
        "field.acceleration_or_force.x": "target_acceleration_x", "field.acceleration_or_force.y": "target_acceleration_y", "field.acceleration_or_force.z": "target_acceleration_z",
        "field.yaw": "target_yaw", "field.yaw_rate": "target_yaw_rate"},
    "mavros-global_position-raw-fix": {**COMMON_HDR, "field.status.status": "gps_status", "field.status.service": "gps_service", "field.latitude": "latitude", "field.longitude": "longitude", "field.altitude": "gps_altitude",
        **get_cov("position_covariance", "gps_position_covariance"), "field.position_covariance_type": "gps_position_covariance_type"},
    "mavros-global_position-raw-gps_vel": {**COMMON_HDR, "field.twist.linear.x": "gps_raw_linear_velocity_x", "field.twist.linear.y": "gps_raw_linear_velocity_y", "field.twist.linear.z": "gps_raw_linear_velocity_z",
        "field.twist.angular.x": "gps_raw_angular_velocity_x", "field.twist.angular.y": "gps_raw_angular_velocity_y", "field.twist.angular.z": "gps_raw_angular_velocity_z"},
    
    # 🟢 NHÓM 4: CẢM BIẾN VẬT LÝ
    "mavros-imu-data": {**COMMON_HDR, "field.orientation.x": "imu_orientation_qx", "field.orientation.y": "imu_orientation_qy", "field.orientation.z": "imu_orientation_qz", "field.orientation.w": "imu_orientation_qw", **get_cov("orientation_covariance", "imu_orientation_covariance"),
        "field.angular_velocity.x": "imu_angular_velocity_x", "field.angular_velocity.y": "imu_angular_velocity_y", "field.angular_velocity.z": "imu_angular_velocity_z", **get_cov("angular_velocity_covariance", "imu_angular_velocity_covariance"),
        "field.linear_acceleration.x": "imu_linear_acceleration_x", "field.linear_acceleration.y": "imu_linear_acceleration_y", "field.linear_acceleration.z": "imu_linear_acceleration_z", **get_cov("linear_acceleration_covariance", "imu_linear_acceleration_covariance")},
    "mavros-wind_estimation": {**COMMON_HDR, "field.twist.linear.x": "estimated_wind_velocity_x", "field.twist.linear.y": "estimated_wind_velocity_y", "field.twist.linear.z": "estimated_wind_velocity_z",
        "field.twist.angular.x": "estimated_wind_angular_velocity_x", "field.twist.angular.y": "estimated_wind_angular_velocity_y", "field.twist.angular.z": "estimated_wind_angular_velocity_z"},
    "mavros-imu-data_raw": {**COMMON_HDR, "field.orientation.x": "imu_raw_orientation_qx", "field.orientation.y": "imu_raw_orientation_qy", "field.orientation.z": "imu_raw_orientation_qz", "field.orientation.w": "imu_raw_orientation_qw", **get_cov("orientation_covariance", "imu_raw_orientation_covariance"),
        "field.angular_velocity.x": "imu_raw_angular_velocity_x", "field.angular_velocity.y": "imu_raw_angular_velocity_y", "field.angular_velocity.z": "imu_raw_angular_velocity_z", **get_cov("angular_velocity_covariance", "imu_raw_angular_velocity_covariance"),
        "field.linear_acceleration.x": "imu_raw_linear_acceleration_x", "field.linear_acceleration.y": "imu_raw_linear_acceleration_y", "field.linear_acceleration.z": "imu_raw_linear_acceleration_z", **get_cov("linear_acceleration_covariance", "imu_raw_linear_acceleration_covariance")},
    "mavros-imu-mag": {**COMMON_HDR, "field.magnetic_field.x": "magnetic_field_x", "field.magnetic_field.y": "magnetic_field_y", "field.magnetic_field.z": "magnetic_field_z", **get_cov("magnetic_field_covariance", "magnetic_field_covariance")},
    "mavros-imu-atm_pressure": {**COMMON_HDR, "field.fluid_pressure": "fluid_pressure", "field.variance": "fluid_pressure_variance"},
    "mavros-imu-temperature": {**COMMON_HDR, "field.temperature": "imu_temperature", "field.variance": "imu_temperature_variance"},

    # 🟣 NHÓM 5: HỆ THỐNG ĐIỆN VÀ TÍN HIỆU RC
    "mavros-battery": {**COMMON_HDR, "field.voltage": "battery_voltage", "field.current": "battery_current", "field.charge": "battery_charge", "field.capacity": "battery_capacity", "field.design_capacity": "battery_design_capacity", 
        "field.percentage": "battery_percentage", "field.power_supply_status": "power_supply_status", "field.power_supply_health": "power_supply_health", "field.power_supply_technology": "power_supply_technology", 
        "field.present": "battery_present", "field.location": "battery_location", "field.serial_number": "battery_serial_number"},
    "mavros-vfr_hud": {**COMMON_HDR, "field.airspeed": "airspeed", "field.groundspeed": "groundspeed", "field.heading": "hud_heading", "field.throttle": "throttle", "field.altitude": "hud_altitude", "field.climb": "climb_rate"},
    "mavros-rc-out": {**COMMON_HDR, "field.channels0": "rc_out_aileron", "field.channels1": "rc_out_elevator", "field.channels2": "rc_out_throttle", "field.channels3": "rc_out_rudder",
        "field.channels4": "rc_out_aux_5", "field.channels5": "rc_out_aux_6", "field.channels6": "rc_out_aux_7", "field.channels7": "rc_out_aux_8"},
    "mavros-rc-in": {**COMMON_HDR, "field.rssi": "rc_input_rssi", "field.channels0": "rc_in_aileron", "field.channels1": "rc_in_elevator", "field.channels2": "rc_in_throttle", "field.channels3": "rc_in_rudder",
        "field.channels4": "rc_in_aux_5", "field.channels5": "rc_in_aux_6", "field.channels6": "rc_in_aux_7", "field.channels7": "rc_in_aux_8"},
    "mavros-setpoint_raw-target_global": {**COMMON_HDR, "field.coordinate_frame": "coordinate_frame", "field.type_mask": "type_mask", "field.latitude": "target_latitude", "field.longitude": "target_longitude", 
        "field.altitude": "target_altitude", "field.velocity.x": "target_velocity_x", "field.velocity.y": "target_velocity_y", "field.velocity.z": "target_velocity_z", 
        "field.acceleration_or_force.x": "target_acceleration_x", "field.acceleration_or_force.y": "target_acceleration_y", "field.acceleration_or_force.z": "target_acceleration_z", 
        "field.yaw": "target_yaw", "field.yaw_rate": "target_yaw_rate"},
    "mavros-state": {**COMMON_HDR, "field.connected": "is_connected", "field.armed": "is_armed", "field.guided": "is_guided", "field.mode": "flight_mode", "field.system_status": "system_status"},
    "mavros-time_reference": {**COMMON_HDR, "field.time_ref": "reference_time_ns", "field.source": "time_source"},
    "mavros-mavlink-from": {**COMMON_HDR, "field.framing_status": "framing_status", "field.magic": "mavlink_magic_byte", "field.len": "payload_length", 
        "field.incompat_flags": "incompatibility_flags", "field.compat_flags": "compatibility_flags", "field.seq": "mavlink_packet_seq", "field.sysid": "mavlink_sys_id", "field.compid": "mavlink_comp_id", 
        "field.msgid": "mavlink_msg_id", "field.checksum": "mavlink_checksum", "field.payload640": "mavlink_payload_64_0", "field.payload641": "mavlink_payload_64_1", "field.payload642": "mavlink_payload_64_2", "field.payload643": "mavlink_payload_64_3"}
}

# Hàm Tự Sinh Map cho `diagnostics.csv`
diag_map = {**COMMON_HDR}
for n in range(5):
    diag_map[f"field.status{n}.level"] = f"diag_level_{n}"
    diag_map[f"field.status{n}.name"] = f"diag_name_{n}"
    diag_map[f"field.status{n}.message"] = f"diag_message_{n}"
    diag_map[f"field.status{n}.hardware_id"] = f"diag_hardware_id_{n}"
    for m in range(26):
        diag_map[f"field.status{n}.values{m}.key"] = f"diag_{n}_val_key_{m}"
        diag_map[f"field.status{n}.values{m}.value"] = f"diag_{n}_val_value_{m}"
COLUMN_MAPPING["diagnostics"] = diag_map

TARGET_FILES = list(COLUMN_MAPPING.keys())
print(f"Đã tải cấu hình ánh xạ cho {len(TARGET_FILES)} loại file dữ liệu!")

Đã tải cấu hình ánh xạ cho 37 loại file dữ liệu!


In [3]:

def extract_flight_metadata(folder_name):
    # Trích xuất metadata từ tên thư mục gốc
    match = re.match(r"^([a-zA-Z0-9]+)_(\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2})_(.*)$", folder_name)
    if not match: return None
    
    aircraft_type, flight_sequence_id, remaining = match.groups()
    
    fault_match = re.match(r"^\d+_(.*)$", remaining)
    sequence_fault_type = fault_match.group(1) if fault_match else remaining
    is_faulty_sequence = 0 if sequence_fault_type in ['no_failure', 'no_ground_truth'] else 1
    
    return {
        "aircraft_type": aircraft_type,
        "flight_sequence_id": flight_sequence_id,
        "sequence_fault_type": sequence_fault_type,
        "is_faulty_sequence": is_faulty_sequence
    }

In [4]:
# %%
def process_single_flight(folder_path, folder_name):
    dataframes = []
    
    # 1. Đọc và lọc toàn bộ các file
    for file_func in TARGET_FILES:
        file_path = os.path.join(folder_path, f"{folder_name}-{file_func}.csv")
        
        if os.path.exists(file_path):
            try:
                df = pd.read_csv(file_path)
                col_map = COLUMN_MAPPING[file_func]
                
                # Chỉ lấy những cột thực sự tồn tại
                existing_cols = [c for c in col_map.keys() if c in df.columns]
                df = df[existing_cols]
                df.rename(columns=col_map, inplace=True)
                
                if 'recording_timestamp' in df.columns:
                    # Gộp mốc thời gian nanosecond thành millisecond để tránh tràn RAM
                    if ROUND_TO_MILLISECOND:
                        df['recording_timestamp'] = (df['recording_timestamp'] // 1000000) * 1000000
                        df = df.groupby('recording_timestamp').last().reset_index()
                    else:
                        df = df.drop_duplicates(subset=['recording_timestamp'])
                    
                    df.set_index('recording_timestamp', inplace=True)
                    
                    # Đổi tên các cột metadata chung để không bị đè mất khi join
                    overlap_cols = ['sequence_number', 'sensor_timestamp', 'frame_id', 'coordinate_frame', 'type_mask', 'child_frame_id']
                    rename_dict = {}
                    for col in overlap_cols:
                        if col in df.columns:
                            # Gắn thêm tên file đằng sau (VD: sequence_number_nav_roll)
                            rename_dict[col] = f"{col}_{file_func.split('-')[-1]}"
                    if rename_dict:
                        df.rename(columns=rename_dict, inplace=True)

                    dataframes.append(df)
            except Exception as e:
                pass 

    if not dataframes:
        return None

    # 2. Gộp tất cả Dataframes lại theo mốc thời gian
    merged_df = pd.concat(dataframes, axis=1, join='outer')
    
    
    merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()].copy()
    
    # --- BẮT ĐẦU ĐOẠN ĐÃ SỬA LỖI ---
    # 3. Nội suy Zero-Order Hold (ffill) cho TẤT CẢ các cột trước.
    # Bước này CỰC KỲ QUAN TRỌNG với Time-Series để duy trì trạng thái cảm biến (ví dụ: GPS 5Hz khớp với IMU 100Hz)
    merged_df = merged_df.ffill()
    
    # 4. Xử lý đoạn đầu chuyến bay (khi các node chưa kịp bắn data)
    # 4.1. Với nhãn lỗi: Trước khi lỗi thì mặc định là 0 (bình thường)
    failure_cols = ['engine_failure_status', 'aileron_failure_status', 'rudder_failure_status', 'elevator_failure_status']
    for col in failure_cols:
        if col in merged_df.columns:
            merged_df[col] = merged_df[col].fillna(0)
            
    # 4.2. Với các cảm biến (GPS, IMU): bfill() để copy giá trị đo được đầu tiên về giây số 0
    # Lúc này bfill() hoàn toàn an toàn vì các cột lỗi đã được xử lý bằng 0 rồi.
    merged_df = merged_df.bfill()
    # --- KẾT THÚC ĐOẠN ĐÃ SỬA LỖI ---
    
    merged_df.reset_index(inplace=True)
    
    # 4. Chèn nhãn phân loại chuyến bay
    metadata = extract_flight_metadata(folder_name)
    if metadata:
        for k, v in metadata.items():
            merged_df[k] = v
            
    return merged_df

In [5]:
# Khai báo đường dẫn
DATA_PATH = "../raw_data" 
folders = [f for f in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, f))]
folders = [f for f in folders if "no_ground_truth" not in f]

all_flights_data = []

print(f"🚀 Bắt đầu xử lý {len(folders)} chuyến bay (Gộp 100% cột - {len(TARGET_FILES)} File/Chuyến)...")

for i, folder in enumerate(folders):
    folder_path = os.path.join(DATA_PATH, folder)
    print(f"Đang xử lý [{i+1:02d}/{len(folders)}]: {folder[:50]}...")
    
    flight_df = process_single_flight(folder_path, folder)
    if flight_df is not None:
        all_flights_data.append(flight_df)

if all_flights_data:
    final_dataset = pd.concat(all_flights_data, ignore_index=True)
    print("\n✅ Xử lý hoàn tất!")
    print(f"Kích thước DataFrame siêu to khổng lồ: {final_dataset.shape[0]} dòng, {final_dataset.shape[1]} cột.")
else:
    print("❌ Không có dữ liệu nào.")

🚀 Bắt đầu xử lý 46 chuyến bay (Gộp 100% cột - 37 File/Chuyến)...
Đang xử lý [01/46]: carbonZ_2018-10-05-15-55-10_engine_failure_with_em...
Đang xử lý [02/46]: carbonZ_2018-10-18-11-03-57_engine_failure_with_em...
Đang xử lý [03/46]: carbonZ_2018-10-18-11-04-00_engine_failure_with_em...
Đang xử lý [04/46]: carbonZ_2018-10-18-11-08-24_no_failure...
Đang xử lý [05/46]: carbonZ_2018-10-05-14-37-22_3_left_aileron_failure...
Đang xử lý [06/46]: carbonZ_2018-10-05-15-52-12_1_no_failure...
Đang xử lý [07/46]: carbonZ_2018-10-18-11-04-35_engine_failure_with_em...
Đang xử lý [08/46]: carbonZ_2018-10-18-11-04-08_2_engine_failure_with_...
Đang xử lý [09/46]: carbonZ_2018-10-18-11-06-06_engine_failure_with_em...
Đang xử lý [10/46]: carbonZ_2018-10-05-16-04-46_engine_failure_with_em...
Đang xử lý [11/46]: carbonZ_2018-10-05-15-52-12_3_engine_failure_with_...
Đang xử lý [12/46]: carbonZ_2018-10-05-15-52-12_2_no_failure...
Đang xử lý [13/46]: carbonZ_2018-10-18-11-04-08_1_engine_failure_with_...
Đang 

In [8]:
output_csv = "../processed/ALFA_Dataset_Full.csv"
output_parquet = "../processed/ALFA_Dataset_Full.parquet"

print("Đang dọn dẹp kiểu dữ liệu để tương thích với Parquet...")

object_columns = final_dataset.select_dtypes(include=['object']).columns
final_dataset[object_columns] = final_dataset[object_columns].astype(str)

# Lưu ra CSV (Có thể tốn vài phút vì file lớn)
print("Đang ghi dữ liệu ra file CSV...")
final_dataset.to_csv(output_csv, index=False)
print(f"✅ Đã lưu xong CSV tại: {output_csv}")

# Lưu ra Parquet
try:
    print("Đang ghi dữ liệu ra file Parquet...")
    # Nén file Parquet (engine='pyarrow') siêu tốc
    final_dataset.to_parquet(output_parquet, engine='pyarrow', index=False)
    print(f"✅ Đã lưu xong Parquet tại: {output_parquet}")
    
    # Tính toán dung lượng thu gọn
    csv_size = os.path.getsize(output_csv) / (1024 * 1024)
    pq_size = os.path.getsize(output_parquet) / (1024 * 1024)
    print(f"\n📊 Báo cáo dung lượng:")
    print(f"- File CSV: {csv_size:.2f} MB")
    print(f"- File Parquet: {pq_size:.2f} MB (Tiết kiệm được {(1 - pq_size/csv_size)*100:.1f}%)")
    
except Exception as e:
    print(f"❌ Có lỗi khi lưu Parquet: {e}")

Đang dọn dẹp kiểu dữ liệu để tương thích với Parquet...
Đang ghi dữ liệu ra file CSV...
✅ Đã lưu xong CSV tại: ../processed/ALFA_Dataset_Full.csv
Đang ghi dữ liệu ra file Parquet...
✅ Đã lưu xong Parquet tại: ../processed/ALFA_Dataset_Full.parquet

📊 Báo cáo dung lượng:
- File CSV: 2641.06 MB
- File Parquet: 29.75 MB (Tiết kiệm được 98.9%)
